# 用户级抽样：抽取跨 7 个月的用户面板（UserValue&Potential 项目）

在 Kaggle 上直接读取 [eCommerce behavior data from multi category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store)（Kaggle 版只有 2019-10 和 2019-11 两个月），**按用户抽样**：

- 每个用户要么**全部月份都在面板里**，要么完全不在（用 `user_id` 的确定性哈希判断，跨文件一致）——保证能跨月追踪同一用户；
- 默认保留 5% 的用户（`KEEP_PER_MILLE = 50`），输出单个 parquet，**只下载这个小面板**，本地不需要全量数据。

**两条路径：**
1. **只有 Kaggle 的两个月**：直接 Run All，得到 2019-10/11 全量面板（已远超本地抽样版）；
2. **要 7 个月**：把数据集页面里 “additional archives”（Dec 2019 – Apr 2020）的**公开直链**填入下方 `EXTRA_URLS`，Notebook 会逐个下载→抽样→删除（峰值磁盘占用最小），7 个月一起进面板。

运行后到右侧 **Output** 面板下载 `panel_7months.parquet`。

In [ ]:
# ═══════════════════════════════════════════════════════════
# 用户级抽样：抽取跨月“用户面板”（2019-10 起，可扩展至 7 个月）
# ═══════════════════════════════════════════════════════════
import glob
import os
import zipfile

import numpy as np
import pandas as pd
import requests

OUT_FILE = '/kaggle/working/panel_7months.parquet'
KEEP_PER_MILLE = 50      # 每千个用户保留 50 个 = 5%（若每月行数不足 10 万可调大，如 100=10%）
HASH_MOD = np.uint64(1000)
HASH_MULT = np.uint64(2654435761)   # Knuth 乘法散列：同一 user_id 在任何文件/月份判断结果一致

COLS = ['event_time', 'event_type', 'product_id', 'category_id',
        'category_code', 'brand', 'price', 'user_id', 'user_session']

EXTRA_URLS = [
    # 可选：填入“additional archives”里 Dec 2019 – Apr 2020 的公开直链，例如：
    # 'https://example.com/2019-Dec.csv',
    'https://data.rees46.com/datasets/marketplace/2019-Dec.csv.gz',
    'https://data.rees46.com/datasets/marketplace/2020-Jan.csv.gz',
    'https://data.rees46.com/datasets/marketplace/2020-Feb.csv.gz',
    'https://data.rees46.com/datasets/marketplace/2020-Mar.csv.gz',
    'https://data.rees46.com/datasets/marketplace/2020-Apr.csv.gz'
]


def find_csv_files(root: str) -> list:
    """递归发现目录下所有 CSV（不硬编码数据集文件夹名）。"""
    out = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith('.csv'):
                out.append(os.path.join(dirpath, fn))
    return sorted(out)


def fetch_extra(urls: list, work_dir: str = '/kaggle/working/extra') -> list:
    """逐个下载补充月文件到 work_dir；若为 zip 则解压出其中的 CSV。"""
    os.makedirs(work_dir, exist_ok=True)
    got = []
    for url in urls:
        name = os.path.basename(url.split('?')[0])
        dest = os.path.join(work_dir, name)
        print(f'下载 {name} ...')
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(dest, 'wb') as fh:
                for block in r.iter_content(1 << 20):
                    fh.write(block)
        if dest.lower().endswith('.zip'):
            with zipfile.ZipFile(dest) as z:
                z.extractall(work_dir)
            os.remove(dest)
        print(f'  完成: {os.path.basename(dest)} {os.path.getsize(dest)/1e9:.2f} GB')
    got = find_csv_files(work_dir)
    return got


def keep_user(uid: pd.Series) -> np.ndarray:
    """只依赖 user_id 数值的确定性抽样：同一用户在所有月份得到相同结论。

    全量原始文件里 user_id 可能被 pandas 读成 float64（存在缺失值时），直接
    to_numpy(dtype=uint64) 会把缺失值转成垃圾大数、破坏哈希，因此先统一转数值。
    """
    numeric = pd.to_numeric(uid, errors='coerce')
    valid = numeric.notna().to_numpy()
    u = numeric.to_numpy(dtype=np.int64).astype(np.uint64)
    h = (u * HASH_MULT) % HASH_MOD
    return valid & (h < np.uint64(KEEP_PER_MILLE))


def sample_file(path: str) -> tuple:
    """流式抽样单个文件，返回 (保留行数, 全量行数, 缺失user_id行数)。"""
    keep_rows = total_rows = bad_id = 0
    kept_chunks = []
    for chunk in pd.read_csv(path, usecols=COLS, chunksize=2_000_000):
        sel = keep_user(chunk['user_id'])
        total_rows += len(chunk)
        keep_rows += int(sel.sum())
        bad_id += int(((~sel) & chunk['user_id'].isna()).sum())
        if sel.any():
            kept_chunks.append(chunk.loc[sel])
    print(f'{os.path.basename(path)}: 全量 {total_rows:,} 行 → 保留 {keep_rows:,} 行 '
          f'(user_id 缺失 {bad_id:,} 行)')
    return kept_chunks, keep_rows, total_rows


# ── 1. 收集文件：Kaggle 挂载的 + 可选直链下载的补充月 ──
files = find_csv_files('/kaggle/input')
if not files:
    print('❌ 未在 /kaggle/input 下找到任何 CSV。当前目录结构如下：')
    for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
        print(f'  {dirpath}/')
        for d in dirnames:
            print(f'    [目录] {d}/')
        for f in filenames[:20]:
            print(f'    [文件] {f}')
    print('\n请检查：① 是否已通过右侧 Add Input 把数据集挂载进来；② 挂载后重新运行。')
    raise SystemExit('未找到 CSV 文件')

if EXTRA_URLS:
    files += fetch_extra(EXTRA_URLS)

print(f'\n共 {len(files)} 个月份文件:')
for f in files:
    print(f'  {os.path.basename(f)}  {os.path.getsize(f) / 1e9:.2f} GB')
if len(files) < 5:
    print(f'\n⚠️ 注意：只有 {len(files)} 个月（期望 7 个月：2019-10 ~ 2020-04）。'
          f'如需 5 个月以上，请把 additional archives 的直链填入 EXTRA_URLS，'
          f'或改用阿里云天池镜像一次性下载后走本地脚本。')

# ── 2. 逐个文件抽样（补充月抽完即删，控制峰值磁盘）──
kept_all = []
total_kept = 0
for f in files:
    chunks, keep_rows, _ = sample_file(f)
    kept_all.extend(chunks)
    total_kept += keep_rows
    if '/kaggle/working/extra/' in f:          # 直链下载的临时文件，抽完删除
        os.remove(f)

if total_kept == 0:
    raise SystemExit('没有任何行被保留！请检查 user_id 列内容（KEEP_PER_MILLE 是否过小 / user_id 是否全为缺失）。')

# ── 3. 合并输出 ──
panel = pd.concat(kept_all, ignore_index=True)
panel['month'] = panel['event_time'].str[:7]        # '2019-10-01 …' → '2019-10'
panel.to_parquet(OUT_FILE, index=False)

print(f'\n面板总行数: {len(panel):,}')
print(f'面板用户数: {panel["user_id"].nunique():,}')
print('\n每月事件数:')
print(panel.groupby('month').size().to_string())
print(f'\n已保存: {OUT_FILE}')